In [1]:
# --- repo bootstrap: make src/ + this domain's config importable, run from repo root ---
# domains/credit_risk/ is mounted only in the credit-risk containers, so the
# domain-agnostic stages physically cannot import domain settings.
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
sys.path.insert(0, str(_ROOT / 'domains' / 'credit_risk'))
os.chdir(_ROOT)

# Aave V3.1 — model features (quality + structure evidence)

Feature-layer evidence for the modeling stage, on the CORRECT frames:
the 2h protocol panel plus the dense **24h** liquidation / user-account rollups
(the old notebook profiled the 53-row 7d frames while labelling them 24h).

In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

import EDA
import adv_validation as adv
import feature_engineering as fe
import model_config as cfg

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

DATA_DIR = Path("transformed_data")
PREVIEW_ROWS = 5

df = pd.read_csv(DATA_DIR / "DF_common_final_1.csv")
df_liq_feat = pd.read_csv(DATA_DIR / "DF_liq_features_24h.csv")
df_user_feat = pd.read_csv(DATA_DIR / "DF_user_features_24h.csv")

for name, fr in [("DF_common_final_1 (2h)", df),
                 ("DF_liq_features_24h", df_liq_feat),
                 ("DF_user_features_24h", df_user_feat)]:
    print(f" {name}: {fr.shape[0]} rows x {fr.shape[1]} cols")

 DF_common_final_1 (2h): 4368 rows x 61 cols
 DF_liq_features_24h: 364 rows x 24 cols
 DF_user_features_24h: 364 rows x 21 cols


In [3]:
# the ONE weight scheme (model_config.CREDIT_RISK_WEIGHTS) routed to each column's home frame
weight_map = dict(cfg.CREDIT_RISK_WEIGHTS)
cols = list(weight_map)

liq_home = set(df_liq_feat.columns)
user_home = set(df_user_feat.columns)

def home_frame(col):
    if col in liq_home:
        return "DF_liq_features_24h", df_liq_feat
    if col in user_home:
        return "DF_user_features_24h", df_user_feat
    return "DF_common_final_1", df

routed = {}
for col in cols:
    fname, fr = home_frame(col)
    routed.setdefault(fname, (fr, []))[1].append(col)

parts = []
for fname, (fr, fcols) in routed.items():
    present = [c for c in fcols if c in fr.columns]
    missing = [c for c in fcols if c not in fr.columns]
    if missing:
        print(f"[warn] {fname} missing: {missing}")
    r = adv.statistical_validation(fr, columns=present, save=False)[
        ["column", "null_pct", "zero_pct", "negative_pct", "cv",
         "p95", "p99", "skewness", "excess_kurtosis",
         "outlier_iqr_pct", "outlier_mad_pct"]].copy()
    r.insert(1, "frame", fname)
    parts.append(r)

quality = pd.concat(parts, ignore_index=True)
quality["weight"] = quality["column"].map(weight_map)
quality = quality.sort_values("weight", ascending=False).reset_index(drop=True)
display(quality)

[warn] DF_common_final_1 missing: ['has_liquidation']


,column,frame,null_pct,zero_pct,negative_pct,cv,p95,p99,skewness,excess_kurtosis,outlier_iqr_pct,outlier_mad_pct,weight
0,ltv_utilization,DF_user_features_24h,0.0000,0.0000,0.0000,0.018578,9.768957e-01,9.789474e-01,-13.328233,220.092730,2.1978,1.0989,9.0
1,distance_to_liquidation,DF_user_features_24h,0.0000,0.0000,0.0000,0.512727,4.692128e-02,5.923854e-02,13.328233,220.092730,2.1978,1.0989,9.0
2,borrow_capacity_utilization,DF_user_features_24h,0.0000,1.0989,0.0000,0.313593,9.428554e-01,9.823077e-01,-0.440365,0.116380,1.0989,0.0000,7.0
3,avg_ltv,DF_user_features_24h,0.0000,0.0000,0.0000,0.099400,8.854623e-01,9.300000e-01,-2.002238,7.339919,8.7912,4.1209,7.0
4,liquidation_rate,DF_liq_features_24h,0.0000,8.5165,0.0000,3.859815,1.685564e-01,9.944312e-01,7.463423,64.761808,14.2857,14.2857,6.0
5,risk_buffer,DF_user_features_24h,0.0000,0.0000,0.0000,0.459420,3.739048e-02,4.574860e-02,12.335008,195.322719,2.7473,1.3736,6.0
6,leverage_indicator,DF_common_final_1,0.0000,0.0000,0.0000,0.959325,9.608501e-01,1.811957e+00,5.194395,64.559060,4.0064,2.5870,5.0
7,overcollateral_margin,DF_user_features_24h,0.0000,0.0000,0.0000,1.208626,6.313090e+07,9.440570e+07,1.999508,4.936308,6.3187,7.4176,4.0
8,borrow_repay_ratio,DF_common_final_1,0.0000,0.0000,0.0000,1.305759,3.476962e+00,8.258049e+00,9.739441,156.298127,9.2262,7.4176,4.0
9,market_stress_index_usd,DF_liq_features_24h,0.0000,8.5165,0.0000,7.023905,1.921545e+00,2.228357e+02,8.936325,87.302296,19.2308,38.1868,3.0


In [4]:
# skew evidence for the split-stage log1p transform, implying heavy-tailed non-negative columns
heavy = quality.loc[(quality["skewness"] > cfg.SKEW_LOG1P_THRESHOLD)
                    & (quality["negative_pct"] == 0)]
print(f" {len(heavy)} of {len(quality)} weighted columns exceed skew "
      f"{cfg.SKEW_LOG1P_THRESHOLD} -> log1p candidates in model_split")
display(heavy[["column", "frame", "weight", "skewness", "excess_kurtosis", "p99"]]
        .reset_index(drop=True))

 31 of 52 weighted columns exceed skew 2.5 -> log1p candidates in model_split


,column,frame,weight,skewness,excess_kurtosis,p99
0,distance_to_liquidation,DF_user_features_24h,9.0,13.328233,220.092730,5.923854e-02
1,liquidation_rate,DF_liq_features_24h,6.0,7.463423,64.761808,9.944312e-01
2,risk_buffer,DF_user_features_24h,6.0,12.335008,195.322719,4.574860e-02
3,leverage_indicator,DF_common_final_1,5.0,5.194395,64.559060,1.811957e+00
4,borrow_repay_ratio,DF_common_final_1,4.0,9.739441,156.298127,8.258049e+00
5,market_stress_index_usd,DF_liq_features_24h,3.0,8.936325,87.302296,2.228357e+02
6,liquidation_severity_usd,DF_liq_features_24h,3.0,7.207527,72.456561,2.617053e+05
7,repayment_discipline,DF_common_final_1,3.0,3.115289,19.543267,1.821143e+00
8,liquidated_collateral_value_usd,DF_liq_features_24h,2.0,7.128183,54.958562,9.388813e+07
9,liquidation_severity_eth,DF_liq_features_24h,2.0,5.759756,39.606409,1.224090e+02


## Column structure — co-movement clusters and tail-risk tiers

Which panel features are redundant (signed-correlation clusters) and which are
tail-wild (kurtosis · Hill α · robust-CV composite etc.). All computed on the 2h panel.

In [5]:
num = fe.numeric_columns(df)
corr_split = fe.split_columns_by_correlation(df, columns=num, n_clusters=4)
coh = fe.cluster_coherence(df, corr_split["groups"])
display(coh)
for gname, gcols in corr_split["groups"].items():
    print(f" {gname}: {len(gcols)} cols -> {gcols[:6]}{' ...' if len(gcols) > 6 else ''}")

,n_cols,mean_within_r
group,,
cluster_1,37.0,0.300436
cluster_2,11.0,0.551055
cluster_3,8.0,0.217525
cluster_4,4.0,0.320022
overall,60.0,0.154239


 cluster_1: 37 cols -> ['flashloan_amount_value_usd', 'flashloan_amount_value_eth', 'avg_flashloan_size_usd', 'avg_flashloan_size_eth', 'leverage_indicator', 'borrow_growth'] ...
 cluster_2: 11 cols -> ['avg_supply_size_usd', 'avg_withdrawal_size_usd', 'withdrawal_amount_value_usd', 'supply_amount_value_usd', 'protocol_turnover_usd', 'supply_amount_value_eth'] ...
 cluster_3: 8 cols -> ['liquidity_growth_rate', 'collateral_usage_rate', 'borrow_repay_ratio', 'net_borrow_demand_usd', 'net_borrow_demand_eth', 'supply_withdrawal_ratio'] ...
 cluster_4: 4 cols -> ['flashloan_fee_rate', 'no_debt_flashloan_ratio', 'repayment_discipline', 'flashloan_user_activity']


In [6]:
tail_split = fe.split_columns_by_tail_risk(df, columns=num)
profile = fe.column_group_profile(df, tail_split["groups"])
display(profile)
for tier, tcols in tail_split["groups"].items():
    print(f" {tier}: {len(tcols)} columns")

,n_cols,mean_excess_kurtosis,mean_hill_tail_index,mean_robust_cv_iqr
stable,20.0,14.058341,1012.124355,0.551215
moderate,20.0,101.169922,2.412044,1.111661
wild,20.0,804.326152,1.834314,200.001188


 stable: 20 columns
 moderate: 20 columns
 wild: 20 columns


## Row structure — balanced market regimes

Three independent tercile splits of the 4 380 buckets (volatility / activity /
whale dominance) with balance checks — regime context for reading model errors.

In [7]:
vol_split = fe.split_by_volatility_regime(df)
act_split = fe.split_by_activity_intensity(df)
whale_split = fe.split_by_whale_dominance(df)

for name, sp in [("volatility", vol_split), ("activity", act_split),
                 ("whale", whale_split)]:
    print(f" {name}: {fe.split_balance(sp['labels'])}")

 volatility:               n    pct
calm       1455  33.33
normal     1455  33.33
turbulent  1455  33.33
 activity:            n    pct
quiet   1473  33.72
active  1444  33.06
peak    1451  33.22
 whale:            n    pct
retail  1456  33.33
mixed   1456  33.33
whale   1456  33.33


In [8]:
# regime evidence: median of key stress columns per volatility regime
evidence_cols = [c for c in ("liquidation_count_24h", "net_liquidity_flow_usd",
                             "borrow_repay_ratio", "protocol_turnover_usd")
                 if c in df.columns]
display(fe.group_stat_table(vol_split["frames"], evidence_cols))

,calm,normal,turbulent
net_liquidity_flow_usd,1.300478e+06,2.029276e+06,1.800246e+06
borrow_repay_ratio,1.031263e+00,1.046869e+00,1.003266e+00
protocol_turnover_usd,2.181820e+08,2.992369e+08,3.678713e+08
